# Random numbers and distributions

> Seeded sampling and probability functions.

In [ ]:
import rustymath as rm

## A persistent generator

`default_rng(seed)` creates one generator whose state advances across calls. A size tuple gives the output shape. Reuse the generator for different distributions:

In [ ]:
rng = rm.default_rng(42)
draws = rng.normal(size=(3, 2))
draws, draws.mean(axis=0), rng.sample(rm.distribution.beta(2, 3), 5)

(Array([[-1.0920367, -0.30040901], [-0.78924237, 0.52985647], [-2.2347094, 1.3224215]], dtype=float64),
 Array([-1.3719962, 0.51728964], dtype=float64),
 Array([0.16794595, 0.31556707, 0.25183041, 0.32702751, 0.60316242], dtype=float64))

`integers` draws from a half-open interval. `choice` and `permutation` operate on leading-axis cells; an integer argument denotes a range. Here the two selected items are whole rows:

In [ ]:
rng.integers(0, 10, size=4), rng.choice(draws, size=2, replace=False)

(Array([5, 3, 0, 4], dtype=int64),
 Array([[-1.0920367, -0.30040901], [-2.2347094, 1.3224215]], dtype=float64))

The generator uses rustymath's shared RNG. It does not reproduce NumPy's random sequences.

`multivariate_normal(mean, cov, size)` correlates ordinary normal draws through a Cholesky factor and a matrix product. It takes one real mean vector and real positive-definite covariance matrix, both without axis keys or names, returning float64 samples with a final component axis. A covariance that cannot be factored follows `cholesky`'s NaN-result policy.

In [ ]:
rng.multivariate_normal([1., 2.], [[4., 1.], [1., 3.]], size=3)

Array([[-0.90930908, 2.008366], [4.0249491, 6.4071364], [-1.7609395, 1.433404]], dtype=float64)

## Probability functions

`rm.distribution` constructors return objects with `pdf`, `cdf`, `quantile` and `sample`. `cdf(x)` is the probability of a value at most `x`. Subtract it from one for an upper tail. The same `seed` gives the same draws:

In [ ]:
n = rm.distribution.normal()
n.cdf([-1.96, 0, 1.96]), 1 - n.cdf(1.96), n.quantile(0.975), n.sample(3, seed=0)

(Array([0.024997895, 0.5, 0.9750021], dtype=float64),
 0.024997895148220484,
 1.9599639845400538,
 Array([0.11623959, -0.78285485, -2.3346353], dtype=float64))

For a discrete distribution, `pdf(k)` gives the probability mass at `k`. With two arrivals expected in an interval, a Poisson law gives the probabilities of zero to three arrivals, the probability of at least four, and the 95th-percentile count:

In [ ]:
arrivals = rm.distribution.poisson(2)
arrivals.pdf([0, 1, 2, 3]), 1 - arrivals.cdf(3), arrivals.quantile(0.95)

(Array([0.13533528, 0.27067057, 0.27067057, 0.18044704], dtype=float64),
 0.14287653950145296,
 5.0)

All constructors share this object interface. `rng.sample(distribution, size)` uses a persistent generator for any of them.

| Constructor | Parameters |
|---|---|
| `normal(mean=0, sd=1)` | Mean and standard deviation |
| `uniform(min=0, max=1)` | Lower and upper limits |
| `beta(a, b)` | Two shapes |
| `cauchy(location=0, scale=1)` | Location and scale |
| `chi_squared(df)` | Degrees of freedom |
| `exponential(rate=1)` | Rate |
| `fisher(df1, df2)` | Numerator and denominator degrees of freedom |
| `gamma(shape, scale)` | Shape and scale |
| `inverse_gamma(shape, scale)` | Shape and scale |
| `laplace(location=0, scale=1)` | Location and scale |
| `logistic(location=0, scale=1)` | Location and scale |
| `log_normal(location=0, scale=1)` | Mean and standard deviation of the logarithm |
| `student_t(df)` | Degrees of freedom |
| `weibull(shape, scale)` | Shape and scale |
| `binomial(n, p)` | Trial count and success probability |
| `poisson(rate)` | Mean count |